In [1]:
pip install dash plotly pandas openpyxl

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.1.1
[notice] To update, run: C:\Users\rajmu\anaconda3\anaconda\python.exe -m pip install --upgrade pip


In [2]:

import pandas as pd
import numpy as np

import dash
from dash import dcc, html
from dash.dependencies import Input, Output

import plotly.express as px

# LOAD DATA

In [3]:
monthly_trend = pd.read_csv(
    "monthly_prescribing_trend.csv"
)

In [4]:
icb_summary = pd.read_csv(
    "icb_prescribing_summary.csv"
)

In [5]:
drug_summary = pd.read_csv(
    "drug_prescribing_summary.csv"
)

In [6]:
dropoff_df = pd.read_csv(
    "dropoff_rate_by_region.csv"
)

# DATE CLEANING

In [7]:
monthly_trend["Month"] = pd.to_datetime(
    monthly_trend["Month"]
)



# DASH APP

In [8]:
app = dash.Dash(__name__)

app.title = "NHS Mental Health Dashboard"

# DROPDOWN OPTIONS

In [9]:

region_options = [
    {"label": region, "value": region}
    for region in sorted(
        icb_summary["ICB_NAME"].dropna().unique()
    )
]

drug_options = [
    {"label": drug, "value": drug}
    for drug in sorted(
        monthly_trend["Drug_Category"].dropna().unique()
    )
]


# APP LAYOUT

In [10]:
app.layout = html.Div([

    html.H1(
        "NHS Mental Health Analytics Dashboard",
        style={
            "textAlign": "center"
        }
    ),

    # ========================================================
    # FILTERS
    # ========================================================

    html.Div([

        html.Div([

            html.Label("Select Drug Category"),

            dcc.Dropdown(
                id="drug_dropdown",
                options=drug_options,
                value="Antidepressant",
                clearable=False
            )

        ], style={
            "width": "30%",
            "display": "inline-block",
            "padding": "10px"
        }),

        html.Div([

            html.Label("Select Region"),

            dcc.Dropdown(
                id="region_dropdown",
                options=region_options,
                value=icb_summary["ICB_NAME"].iloc[0],
                clearable=False
            )

        ], style={
            "width": "40%",
            "display": "inline-block",
            "padding": "10px"
        })

    ]),

    # ========================================================
    # KPI CARDS
    # ========================================================

    html.Div([

        html.Div([
            html.H3("Total Prescriptions"),
            html.H2(id="kpi_total_items")
        ], className="card"),

        html.Div([
            html.H3("Total Cost"),
            html.H2(id="kpi_total_cost")
        ], className="card"),

        html.Div([
            html.H3("Drop-off Rate"),
            html.H2(id="kpi_dropoff")
        ], className="card"),

    ], style={
        "display": "flex",
        "justifyContent": "space-around",
        "margin": "20px"
    }),

    # ========================================================
    # CHARTS
    # ========================================================

    dcc.Graph(id="monthly_trend_chart"),

    dcc.Graph(id="cost_trend_chart"),

    dcc.Graph(id="top_medicines_chart"),

    dcc.Graph(id="forecast_chart"),

    dcc.Graph(id="funnel_chart"),

])




# ============================================================
# FORECAST, FUNNEL AND DROPOFF DATA
# ============================================================

forecast_df = monthly_trend.copy()

forecast_df["Forecast_Total_Items"] = (
    forecast_df["Total_Items"] * 1.05
)

funnel_df = pd.DataFrame({
    "Stage": [
        "Prescribed",
        "Dispensed",
        "Completed Treatment"
    ],
    "Count": [
        monthly_trend["Total_Items"].sum(),
        monthly_trend["Total_Items"].sum() * 0.85,
        monthly_trend["Total_Items"].sum() * 0.65
    ]
})

dropoff_df = pd.DataFrame({
    "Dropoff_Rate": [35.0]
})


# ============================================================
# CALLBACKS
# ============================================================

@app.callback(
    [
        Output("monthly_trend_chart", "figure"),
        Output("cost_trend_chart", "figure"),
        Output("top_medicines_chart", "figure"),
        Output("forecast_chart", "figure"),
        Output("funnel_chart", "figure"),
        Output("kpi_total_items", "children"),
        Output("kpi_total_cost", "children"),
        Output("kpi_dropoff", "children"),
    ],
    [
        Input("drug_dropdown", "value"),
        Input("region_dropdown", "value")
    ]
)

def update_dashboard(selected_drug, selected_region):

    # -----------------------------
    # Filter monthly trend
    # -----------------------------

    filtered_monthly = monthly_trend[
        monthly_trend["Drug_Category"] == selected_drug
    ].copy()

    # -----------------------------
    # Filter ICB summary by region
    # -----------------------------

    filtered_icb = icb_summary[
        (icb_summary["Drug_Category"] == selected_drug) &
        (icb_summary["ICB_NAME"] == selected_region)
    ].copy()

    # -----------------------------
    # Monthly trend chart
    # -----------------------------

    fig_monthly = px.line(
        filtered_monthly,
        x="Month",
        y="Total_Items",
        title=f"{selected_drug} Monthly Prescribing Trend",
        markers=True
    )

    # -----------------------------
    # Cost trend chart
    # -----------------------------

    fig_cost = px.line(
        filtered_monthly,
        x="Month",
        y="Total_Cost",
        title=f"{selected_drug} Cost Trend",
        markers=True
    )

    # -----------------------------
    # Top medicines chart
    # -----------------------------

    filtered_drugs = drug_summary[
        drug_summary["Drug_Category"] == selected_drug
    ].sort_values(
        "Total_Items",
        ascending=False
    ).head(15)

    fig_drugs = px.bar(
        filtered_drugs,
        x="BNF_CHEMICAL_SUBSTANCE",
        y="Total_Items",
        title=f"Top Medicines - {selected_drug}"
    )

    fig_drugs.update_layout(
        xaxis_tickangle=-45
    )

    # -----------------------------
    # Forecast chart
    # -----------------------------

    filtered_forecast = forecast_df[
        forecast_df["Drug_Category"] == selected_drug
    ].copy()

    if len(filtered_forecast) > 0:
        fig_forecast = px.line(
            filtered_forecast,
            x="Month",
            y="Forecast_Total_Items",
            title=f"{selected_drug} 3-Month Forecast",
            markers=True
        )
    else:
        fig_forecast = px.line(
            title="Forecast data not available"
        )

    # -----------------------------
    # Funnel chart
    # -----------------------------

    fig_funnel = px.funnel(
        funnel_df,
        x="Count",
        y="Stage",
        title="Treatment Funnel"
    )

    # -----------------------------
    # KPI values
    # -----------------------------

    if len(filtered_icb) > 0:
        total_items = filtered_icb["Total_Items"].sum()
        total_cost = filtered_icb["Total_Cost"].sum()
    else:
        total_items = filtered_monthly["Total_Items"].sum()
        total_cost = filtered_monthly["Total_Cost"].sum()

    if "Dropoff_Rate" in dropoff_df.columns:
        dropoff = dropoff_df["Dropoff_Rate"].mean()
    else:
        dropoff = 0

    return (
        fig_monthly,
        fig_cost,
        fig_drugs,
        fig_forecast,
        fig_funnel,
        f"{total_items:,.0f}",
        f"£{total_cost:,.2f}",
        f"{dropoff:.2f}%"
    )

# ============================================================
# RUN APP
# ============================================================

if __name__ == "__main__":
    app.run(
        debug=False,
        port=8050,
        jupyter_mode="external"
    )

Dash app running on http://127.0.0.1:8050/
